## GOOGLE COLAB GPU CHECK & RUNTIME VALIDATION

In [15]:
import subprocess, sys, os, platform

print("=" * 60)
print("  FINE-TUNING NOTEBOOK — Qwen2.5-0.5B Medical Domain")
print("  Tech: Unsloth + QLoRA + Replay Buffer")
print("=" * 60)

# Check GPU
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                         "--format=csv,noheader"], capture_output=True, text=True)
if result.returncode == 0:
    gpu_info = result.stdout.strip()
    print(f"\n✅ GPU Detected: {gpu_info}")
else:
    print("\n⚠️  WARNING: No GPU detected! Go to Runtime → Change Runtime Type → T4 GPU")
    print("   This notebook REQUIRES a GPU to run in reasonable time.")

# Check CUDA
import torch
print(f"\n🔦 PyTorch Version : {torch.__version__}")
print(f"🔦 CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🔦 GPU Name        : {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🔦 VRAM            : {total_mem:.2f} GB")
    if total_mem < 10:
        print("⚠️  Less than 10GB VRAM — consider using an even smaller model.")
    else:
        print("✅ VRAM sufficient for Qwen2.5-0.5B with QLoRA (needs ~4-6 GB)")

print(f"\n🐍 Python Version  : {sys.version.split()[0]}")
print(f"💻 Platform        : {platform.system()} {platform.machine()}")

  FINE-TUNING NOTEBOOK — Qwen2.5-0.5B Medical Domain
  Tech: Unsloth + QLoRA + Replay Buffer

✅ GPU Detected: Tesla T4, 15360 MiB, 580.82.07

🔦 PyTorch Version : 2.10.0+cu128
🔦 CUDA Available  : True
🔦 GPU Name        : Tesla T4
🔦 VRAM            : 15.64 GB
✅ VRAM sufficient for Qwen2.5-0.5B with QLoRA (needs ~4-6 GB)

🐍 Python Version  : 3.12.13
💻 Platform        : Linux x86_64


## ELF-RESOLVING DEPENDENCY INSTALLATION

In [16]:
# ============================================================
# This installs the correct, compatible versions automatically.
# Safe to re-run — skips already-installed packages.
# ============================================================
import subprocess, sys, importlib, json
from packaging import version

def run_pip(*args, quiet=True):
    cmd = [sys.executable, "-m", "pip", "install"] + list(args)
    if quiet:
        cmd.append("-q")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ⚠️  Install warning: {result.stderr[-300:]}")
    return result.returncode == 0

def is_installed(pkg, min_version=None):
    try:
        mod = importlib.import_module(pkg.replace("-", "_"))
        if min_version:
            v = getattr(mod, "__version__", "0.0.0")
            return version.parse(v) >= version.parse(min_version)
        return True
    except ImportError:
        return False

print("📦 Installing / verifying dependencies...")
print("   (This may take 3-5 minutes on first run)\n")

# Core packages with minimum versions
PACKAGES = {
    "unsloth":        ("unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git", "2024.1"),
    "transformers":   ("transformers>=4.47.0", "4.47.0"),
    "peft":           ("peft>=0.14.0", "0.14.0"),
    "trl":            ("trl>=0.12.0", "0.12.0"),
    "accelerate":     ("accelerate>=1.2.0", "1.2.0"),
    "bitsandbytes":   ("bitsandbytes>=0.44.0", "0.44.0"),
    "datasets":       ("datasets>=3.0.0", "3.0.0"),
    "evaluate":       ("evaluate>=0.4.2", "0.4.2"),
    "rouge_score":    ("rouge-score>=0.1.2", "0.1.2"),
    "nltk":           ("nltk>=3.8", "3.8"),
    "plotly":         ("plotly>=5.20.0", "5.20.0"),
    "rich":           ("rich>=13.0.0", "13.0.0"),
    "sentencepiece":  ("sentencepiece>=0.2.0", "0.2.0"),
    "xformers":       ("xformers", None),
}

# Special install: unsloth needs to be first
print("  [1/2] Installing Unsloth (optimized fine-tuning engine)...")
run_pip("unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git", quiet=False)

print("\n  [2/2] Installing remaining packages...")
for name, (pkg_spec, min_ver) in PACKAGES.items():
    if name == "unsloth":
        continue
    if not is_installed(name, min_ver):
        print(f"    Installing {name}...")
        run_pip(pkg_spec)
    else:
        print(f"    ✅ {name} already installed")

# Download NLTK data for BLEU
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("\n✅ All dependencies ready!")
print("   Restart runtime if this is a fresh install: Runtime → Restart Session")

# Final version report
import importlib.metadata

def get_version(pkg):
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return "not found"

print("\n📋 Version Report:")
print(f"   transformers : {get_version('transformers')}")
print(f"   peft         : {get_version('peft')}")
print(f"   trl          : {get_version('trl')}")
print(f"   datasets     : {get_version('datasets')}")
print(f"   unsloth      : {get_version('unsloth')}")

📦 Installing / verifying dependencies...
   (This may take 3-5 minutes on first run)

  [1/2] Installing Unsloth (optimized fine-tuning engine)...

  [2/2] Installing remaining packages...
    ✅ transformers already installed
    ✅ peft already installed
    ✅ trl already installed
    ✅ accelerate already installed
    ✅ bitsandbytes already installed
    ✅ datasets already installed
    ✅ evaluate already installed
    Installing rouge_score...
    ✅ nltk already installed
    ✅ plotly already installed
    Installing rich...
    ✅ sentencepiece already installed
    ✅ xformers already installed

✅ All dependencies ready!
   Restart runtime if this is a fresh install: Runtime → Restart Session

📋 Version Report:
   transformers : 5.5.0
   peft         : 0.19.1
   trl          : 0.24.0
   datasets     : 4.3.0
   unsloth      : 2026.5.2


## LOAD BASE MODEL & VISUALIZE ARCHITECTURE
### We load Qwen2.5-0.5B-Instruct with 4-bit quantization via Unsloth for memory efficiency on free Colab T4 GPU.

In [17]:
import warnings
# Suppress Transformers internal deprecation warnings (library-level, not code)
warnings.filterwarnings("ignore", message=".*AttentionMaskConverter.*", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*attention mask.*deprecated.*", category=FutureWarning)

In [18]:
import unsloth
from unsloth import FastLanguageModel
import torch
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ---- Config ----
MODEL_NAME    = "unsloth/Qwen2.5-0.5B-Instruct"
MAX_SEQ_LEN   = 1024       # context window for training
LOAD_IN_4BIT  = True       # 4-bit quantization saves ~75% VRAM
DTYPE         = None       # auto-detect (bfloat16 on Ampere+, float16 on T4)

print(f"🔄 Loading base model: {MODEL_NAME}")
print(f"   4-bit quantization  : {LOAD_IN_4BIT}")
print(f"   Max sequence length : {MAX_SEQ_LEN} tokens")
print("   (May take 1-2 minutes on first download)\n")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
)

print("\n✅ Model loaded successfully!")
print(f"   Model class  : {model.__class__.__name__}")
print(f"   Device       : {next(model.parameters()).device}")

# After loading model & tokenizer, explicitly set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    model.config.pad_token_id = tokenizer.eos_token_id

# Clear the baked-in max_length from the model's GenerationConfig
# so max_new_tokens in model.generate() never conflicts with it
model.generation_config.max_length = None


# ---- Count Parameters ----
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Parameter Count:")
print(f"   Total parameters     : {total_params:,}")
print(f"   Trainable parameters : {trainable_params:,}")

# ---- Visualize Model Architecture Layers ----
# Extract layer names & parameter counts
layer_data = []
for name, param in model.named_parameters():
    layer_data.append({
        "name"   : name,
        "params" : param.numel(),
        "shape"  : str(list(param.shape)),
        "dtype"  : str(param.dtype),
    })

# Group by top-level module
from collections import defaultdict
module_sizes = defaultdict(int)
for d in layer_data:
    top_module = d["name"].split(".")[0]
    module_sizes[top_module] += d["params"]

# Pie chart of parameter distribution
labels = list(module_sizes.keys())
values = list(module_sizes.values())

fig = go.Figure(data=[go.Pie(
    labels=labels,
    values=values,
    hole=0.45,
    textinfo="label+percent",
    hovertemplate="<b>%{label}</b><br>Params: %{value:,}<br>Share: %{percent}<extra></extra>",
    marker=dict(
        colors=px.colors.qualitative.Set3[:len(labels)],
        line=dict(color="white", width=2)
    )
)])

fig.update_layout(
    title=dict(
        text=f"<b>{MODEL_NAME}</b> — Parameter Distribution by Module",
        x=0.5, font=dict(size=18)
    ),
    annotations=[dict(
        text=f"<b>{total_params/1e6:.0f}M</b><br>params",
        x=0.5, y=0.5, font_size=16,
        showarrow=False
    )],
    height=500,
    showlegend=True,
    legend=dict(orientation="h", y=-0.1),
    paper_bgcolor="white",
    plot_bgcolor="white",
)
fig.show()

# ---- Test base model generation (BEFORE fine-tuning) ----
print("\n🧪 Base Model — Naive Medical Q&A Test (BEFORE fine-tuning)")
print("   Question: What are the symptoms of Type 2 Diabetes?")
print("-" * 60)

FastLanguageModel.for_inference(model)
test_prompt = """<|im_start|>system
You are a medical education assistant. Answer the question clearly and accurately.
<|im_end|>
<|im_start|>user
What are the symptoms of Type 2 Diabetes?
<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        max_length=None,        # explicitly clear the model's default max_length
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )
base_response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"\n📝 Base Model Response:\n{base_response}")
print("-" * 60)
print("\n[NOTE] Save this response — we'll compare with fine-tuned output later!")

# Store for comparison
BASE_MODEL_SAMPLE_RESPONSE = base_response
BASE_SAMPLE_QUESTION = "What are the symptoms of Type 2 Diabetes?"

🔄 Loading base model: unsloth/Qwen2.5-0.5B-Instruct
   4-bit quantization  : True
   Max sequence length : 1024 tokens
   (May take 1-2 minutes on first download)

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.

✅ Model loaded successfully!
   Model class  : Qwen2ForCausalLM
   Device       : cuda:0

📊 Parameter Count:
   Total parameters     : 342,185,856
   Trainable parameters : 54,204,288



🧪 Base Model — Naive Medical Q&A Test (BEFORE fine-tuning)
   Question: What are the symptoms of Type 2 Diabetes?
------------------------------------------------------------

📝 Base Model Response:
The symptoms of type 2 diabetes can vary from person to person, but some common symptoms include:

1. Increased thirst and urination
2. Fatigue or weakness
3. Excessive hunger (preoccupation with eating)
4. Varying levels of fatigue
5. Skin rash or itching
6. Nausea or vomiting
7. Increased hunger after smaller meals than usual
8. Dizziness, blurred vision, or unexplained tiredness
9. Changes in skin color

It's important to note that these symptoms do not necessarily indicate the onset of diabetes; they may be caused by various other conditions. If you suspect you have diabetes, it is recommended to consult a healthcare provider for proper diagnosis and management.
------------------------------------------------------------

[NOTE] Save this response — we'll compare with fine-tuned outpu

## LOAD & EXPLORE MEDICAL DATASET
### Dataset: medalpaca/medical_meadow_medical_flashcards
#### Small, clean, domain-specific medical Q&A pairs.
#### Perfect for showing fine-tuning impact on a free Colab.

In [19]:
from datasets import load_dataset
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import re

print("📥 Loading Medical Meadow Flashcards dataset...")
raw_dataset = load_dataset("medalpaca/medical_meadow_medical_flashcards", split="train")

print(f"\n✅ Dataset loaded!")
print(f"   Total samples : {len(raw_dataset)}")
print(f"   Features      : {raw_dataset.features}")
print(f"\n📋 Sample (first 3):")
for i, ex in enumerate(raw_dataset.select(range(3))):
    print(f"\n  [{i+1}] Q: {ex['input'][:100]}...")
    print(f"       A: {ex['output'][:120]}...")

# ---- Basic Stats ----
df = raw_dataset.to_pandas()

# Rename columns for clarity
df = df.rename(columns={"input": "question", "output": "answer"})

# Token length proxy (word count * 1.3)
df["q_len"]   = df["question"].str.split().str.len()
df["a_len"]   = df["answer"].str.split().str.len()
df["total_len"] = df["q_len"] + df["a_len"]

print(f"\n📊 Length Statistics:")
print(f"   Question avg words : {df['q_len'].mean():.1f}  (max: {df['q_len'].max()})")
print(f"   Answer avg words   : {df['a_len'].mean():.1f}  (max: {df['a_len'].max()})")
print(f"   Total avg words    : {df['total_len'].mean():.1f}")

# ---- Visualizations ----
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Question Length Distribution (words)",
        "Answer Length Distribution (words)",
        "Q vs A Length Scatter",
        "Top 20 Medical Terms in Questions"
    ],
    specs=[[{"type": "histogram"}, {"type": "histogram"}],
           [{"type": "scatter"},   {"type": "bar"}]]
)

# 1. Question length histogram
fig.add_trace(
    go.Histogram(x=df["q_len"].clip(upper=100), nbinsx=50,
                 marker_color="#4ECDC4", opacity=0.85,
                 name="Question Length", showlegend=True),
    row=1, col=1
)

# 2. Answer length histogram
fig.add_trace(
    go.Histogram(x=df["a_len"].clip(upper=200), nbinsx=50,
                 marker_color="#FF6B6B", opacity=0.85,
                 name="Answer Length", showlegend=True),
    row=1, col=2
)

# 3. Scatter Q vs A length (sample 1000)
sample = df.sample(min(1000, len(df)), random_state=42)
fig.add_trace(
    go.Scatter(
        x=sample["q_len"], y=sample["a_len"],
        mode="markers", opacity=0.4,
        marker=dict(size=4, color="#45B7D1", line=dict(width=0)),
        name="Q vs A", showlegend=False,
    ),
    row=2, col=1
)

# 4. Top medical keywords
from collections import Counter
all_words = " ".join(df["question"].tolist()).lower()
stop_words = {"the","a","an","of","in","is","to","and","or","for","with","by",
              "are","be","was","what","which","that","this","it","its","as",
              "can","how","does","do","when","where","why","not","from","at"}
words = [w for w in re.findall(r"\b[a-z]{4,}\b", all_words) if w not in stop_words]
top_words = Counter(words).most_common(20)
tw_df = pd.DataFrame(top_words, columns=["term", "count"])

fig.add_trace(
    go.Bar(
        x=tw_df["count"], y=tw_df["term"],
        orientation="h",
        marker_color=px.colors.sequential.Viridis_r[:20],
        name="Term Freq", showlegend=False,
    ),
    row=2, col=2
)

fig.update_layout(
    title_text="<b>Medical Flashcards Dataset — Exploratory Analysis</b>",
    title_x=0.5, title_font_size=18,
    height=750,
    paper_bgcolor="white",
    plot_bgcolor="#FAFAFA",
)
fig.update_xaxes(gridcolor="#EEE")
fig.update_yaxes(gridcolor="#EEE")
fig.show()

# ---- Filter to fit free Colab training budget ----
# Keep samples where total word count is <= 200 (reasonable seq length)
df_filtered = df[df["total_len"] <= 200].reset_index(drop=True)
print(f"\n🔧 After filtering long samples (<=200 words total):")
print(f"   Samples remaining : {len(df_filtered)}")
print(f"   Training budget   : 1000 samples (fast run on free Colab)")

# Stratified sample for training
TRAIN_SIZE = 1000
VAL_SIZE   = 150

df_filtered = df_filtered.sample(frac=1, random_state=42).reset_index(drop=True)
df_train    = df_filtered.iloc[:TRAIN_SIZE]
df_val      = df_filtered.iloc[TRAIN_SIZE:TRAIN_SIZE + VAL_SIZE]

print(f"\n   Train set : {len(df_train)} samples")
print(f"   Val set   : {len(df_val)} samples")
print("\n✅ Dataset ready for training!")

# Store globally
MEDICAL_TRAIN_DF = df_train
MEDICAL_VAL_DF   = df_val

📥 Loading Medical Meadow Flashcards dataset...

✅ Dataset loaded!
   Total samples : 33955
   Features      : {'input': Value('string'), 'output': Value('string'), 'instruction': Value('string')}

📋 Sample (first 3):

  [1] Q: What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?...
       A: Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels....

  [2] Q: What leads to genitourinary syndrome of menopause (atrophic vaginitis)?...
       A: Low estradiol production leads to genitourinary syndrome of menopause (atrophic vaginitis)....

  [3] Q: What does low REM sleep latency and experiencing hallucinations/sleep paralysis suggest?...
       A: Low REM sleep latency and experiencing hallucinations/sleep paralysis suggests narcolepsy....

📊 Length Statistics:
   Question avg words : 14.6  (max: 62)
   Answer avg words   : 53.5  (max: 245)
   Total avg words    : 68.1



🔧 After filtering long samples (<=200 words total):
   Samples remaining : 33859
   Training budget   : 1000 samples (fast run on free Colab)

   Train set : 1000 samples
   Val set   : 150 samples

✅ Dataset ready for training!


## BASELINE EVALUATION — BEFORE FINE-TUNING
### We measure ROUGE-1, ROUGE-2, ROUGE-L, BLEU, and Perplexity on the BASE model (before any training) to establish a clear before/after comparison later.

In [20]:
import evaluate
import torch
import numpy as np
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

# Load metrics
rouge_metric = evaluate.load("rouge")
bleu_metric  = evaluate.load("bleu")

# ---- Build Chat Prompt for Inference ----
def build_prompt(question: str, include_answer: bool = False, answer: str = "") -> str:
    prompt = (
        f"<|im_start|>system\n"
        f"You are a medical education assistant. "
        f"Answer the following medical question clearly and accurately.\n"
        f"<|im_end|>\n"
        f"<|im_start|>user\n{question}\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    if include_answer:
        prompt += answer + "<|im_end|>"
    return prompt

# ---- Generate Predictions on Validation Set ----
def evaluate_model(model, tokenizer, eval_df, num_samples=50, max_new_tokens=120, label=""):
    """Run generation on eval_df and return ROUGE + BLEU + perplexity."""
    FastLanguageModel.for_inference(model)

    predictions, references = [], []
    perplexities = []

    eval_sample = eval_df.sample(min(num_samples, len(eval_df)), random_state=42)

    for _, row in tqdm(eval_sample.iterrows(), total=len(eval_sample),
                       desc=f"Evaluating [{label}]"):
        question = row["question"]
        answer   = row["answer"]
        prompt   = build_prompt(question)

        inputs = tokenizer(prompt, return_tensors="pt",
                           truncation=True, max_length=512).to("cuda")

        # Generation
        with torch.no_grad():
            gen_out = model.generate(
                **inputs,
                max_length=None,
                max_new_tokens=max_new_tokens,
                do_sample=False,         # greedy for reproducible eval
                temperature=1.0,
                repetition_penalty=1.15,
                pad_token_id=tokenizer.eos_token_id,
            )
        pred = tokenizer.decode(
            gen_out[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        ).strip()

        predictions.append(pred)
        references.append(answer)

        # Perplexity — compute on ground-truth continuation
        full_text   = build_prompt(question, include_answer=True, answer=answer)
        enc         = tokenizer(full_text, return_tensors="pt",
                                truncation=True, max_length=MAX_SEQ_LEN).to("cuda")
        with torch.no_grad():
            loss = model(**enc, labels=enc["input_ids"]).loss
        perplexities.append(torch.exp(loss).item())

    # ROUGE
    rouge_scores = rouge_metric.compute(
        predictions=predictions, references=references, use_stemmer=True
    )

    # BLEU — tokenize words
    bleu_preds = [p.split() for p in predictions]
    bleu_refs  = [[r.split()] for r in references]
    bleu_result = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])

    return {
        "rouge1"      : rouge_scores["rouge1"],
        "rouge2"      : rouge_scores["rouge2"],
        "rougeL"      : rouge_scores["rougeL"],
        "bleu"        : bleu_result["bleu"],
        "perplexity"  : np.mean(perplexities),
        "predictions" : predictions,
        "references"  : references,
    }

print("📊 Running BASELINE evaluation on base model (BEFORE fine-tuning)...")
print("   Using 50 validation samples for speed on free Colab.\n")

BASELINE_SCORES = evaluate_model(
    model, tokenizer, MEDICAL_VAL_DF,
    num_samples=50, max_new_tokens=120, label="BASE MODEL"
)

print("\n" + "=" * 55)
print("  📋 BASELINE METRICS (Before Fine-Tuning)")
print("=" * 55)
print(f"  ROUGE-1     : {BASELINE_SCORES['rouge1']:.4f}")
print(f"  ROUGE-2     : {BASELINE_SCORES['rouge2']:.4f}")
print(f"  ROUGE-L     : {BASELINE_SCORES['rougeL']:.4f}")
print(f"  BLEU        : {BASELINE_SCORES['bleu']:.4f}")
print(f"  Perplexity  : {BASELINE_SCORES['perplexity']:.2f}")
print("=" * 55)
print("\n⚠️  These are LOW scores — expected! The base model has NOT")
print("   been fine-tuned on medical content yet.")
print("   After fine-tuning, all metrics should improve significantly.\n")

# Sample output preview
print("🔍 Sample Base Model Output:")
print(f"   Q: {MEDICAL_VAL_DF.iloc[0]['question'][:80]}...")
print(f"   Pred: {BASELINE_SCORES['predictions'][0][:120]}...")
print(f"   Ref:  {BASELINE_SCORES['references'][0][:120]}...")

📊 Running BASELINE evaluation on base model (BEFORE fine-tuning)...
   Using 50 validation samples for speed on free Colab.



Evaluating [BASE MODEL]:   0%|          | 0/50 [00:00<?, ?it/s]


  📋 BASELINE METRICS (Before Fine-Tuning)
  ROUGE-1     : 0.3122
  ROUGE-2     : 0.1300
  ROUGE-L     : 0.2165
  BLEU        : 0.0682
  Perplexity  : 62.11

⚠️  These are LOW scores — expected! The base model has NOT
   been fine-tuned on medical content yet.
   After fine-tuning, all metrics should improve significantly.

🔍 Sample Base Model Output:
   Q: What is the specific scenario in which intrapleural pressure may be positive (re...
   Pred: No, the effects of Parathyroid Hormone (PTH) on calcium (Ca) reabsorption in bones do not primarily involve an increase ...
   Ref:  No, PTH also has a phosphaturic action on the kidney which is important because the calcium released from bone would com...


## PREPARE TRAINING DATA + ANTI-FORGETTING REPLAY BUFFER
### CATASTROPHIC FORGETTING PREVENTION:
#### We mix ~10% general instruction data into training. This is the "Rehearsal / Replay Buffer" strategy — proven effective for domain fine-tuning without losing general capabilities (arXiv 2501.13669, 2603.09892).

In [21]:
from datasets import Dataset, concatenate_datasets
import pandas as pd

print("🛡️  Setting up Anti-Forgetting Replay Buffer...")
print("   Strategy: 10% general instruction replay mixed into training data\n")

# ---- Format medical data into chat template ----
def format_medical_sample(row):
    """Format a medical Q&A into Qwen2.5 chat template."""
    return {
        "text": (
            f"<|im_start|>system\n"
            f"You are a medical education assistant. "
            f"Answer the following medical question clearly and accurately.\n"
            f"<|im_end|>\n"
            f"<|im_start|>user\n{row['question']}\n<|im_end|>\n"
            f"<|im_start|>assistant\n{row['answer']}<|im_end|>"
        )
    }

medical_samples = [format_medical_sample(row) for _, row in MEDICAL_TRAIN_DF.iterrows()]
medical_hf = Dataset.from_list(medical_samples)
print(f"✅ Medical training samples formatted : {len(medical_hf)}")

# ---- Load small general instruction replay data ----
# We use a small slice of UltraChat-200k (general chat instructions)
# to remind the model about general capabilities
print("\n📥 Loading general replay buffer (UltraChat-200k small slice)...")

try:
    ultrachat_raw = load_dataset(
        "HuggingFaceH4/ultrachat_200k",
        split="train_sft",
        streaming=False
    )
    # Take a small sample — 10% of medical training size
    REPLAY_SIZE = int(TRAIN_SIZE * 0.10)
    replay_raw  = ultrachat_raw.shuffle(seed=42).select(range(min(REPLAY_SIZE, len(ultrachat_raw))))

    def format_ultrachat(example):
        """Format UltraChat messages to Qwen2.5 chat template."""
        messages = example.get("messages", [])
        text = ""
        for msg in messages[:4]:   # limit to 4 turns to keep it short
            role    = msg.get("role", "user")
            content = msg.get("content", "")[:300]    # clip long messages
            if role == "system":
                text += f"<|im_start|>system\n{content}\n<|im_end|>\n"
            elif role == "user":
                text += f"<|im_start|>user\n{content}\n<|im_end|>\n"
            elif role == "assistant":
                text += f"<|im_start|>assistant\n{content}<|im_end|>\n"
        return {"text": text.strip()}

    replay_formatted = replay_raw.map(format_ultrachat, remove_columns=replay_raw.column_names)
    replay_formatted = replay_formatted.filter(lambda x: len(x["text"]) > 50)

    print(f"✅ General replay samples loaded : {len(replay_formatted)}")
    USE_REPLAY = True

except Exception as e:
    print(f"⚠️  Could not load UltraChat (may be a network issue): {e}")
    print("   Continuing WITHOUT replay buffer — catastrophic forgetting risk increases!")
    USE_REPLAY = False

# ---- Merge datasets ----
if USE_REPLAY:
    combined_dataset = concatenate_datasets([medical_hf, replay_formatted])
    combined_dataset = combined_dataset.shuffle(seed=42)
    print(f"\n✅ Combined training set: {len(combined_dataset)} samples")
    print(f"   Medical       : {len(medical_hf)} ({len(medical_hf)/len(combined_dataset)*100:.1f}%)")
    print(f"   General Replay: {len(replay_formatted)} ({len(replay_formatted)/len(combined_dataset)*100:.1f}%)")
else:
    combined_dataset = medical_hf.shuffle(seed=42)
    print(f"\n⚠️  Training on medical data only: {len(combined_dataset)} samples")

TRAIN_DATASET = combined_dataset

# ---- Tokenize and filter by length ----
def tokenize_and_check(example):
    tokens = tokenizer(example["text"], truncation=False)
    example["token_count"] = len(tokens["input_ids"])
    return example

print(f"\n🔄 Tokenizing and filtering by max seq length ({MAX_SEQ_LEN})...")
TRAIN_DATASET = TRAIN_DATASET.map(tokenize_and_check, desc="Tokenizing")
before_filter = len(TRAIN_DATASET)
TRAIN_DATASET = TRAIN_DATASET.filter(lambda x: x["token_count"] <= MAX_SEQ_LEN - 32)
print(f"   Kept {len(TRAIN_DATASET)} / {before_filter} samples (within {MAX_SEQ_LEN} token limit)")

print("\n✅ Training data pipeline complete!")
print(f"   Final training set size : {len(TRAIN_DATASET)}")
print("\n📌 WHY replay buffer prevents catastrophic forgetting:")
print("   - Fine-tuning only on medical data → model 'overwrites' general weights")
print("   - 10% general examples during training keep general capabilities alive")
print("   - This is the 'Rehearsal' strategy from continual learning literature")

🛡️  Setting up Anti-Forgetting Replay Buffer...
   Strategy: 10% general instruction replay mixed into training data

✅ Medical training samples formatted : 1000

📥 Loading general replay buffer (UltraChat-200k small slice)...
✅ General replay samples loaded : 100

✅ Combined training set: 1100 samples
   Medical       : 1000 (90.9%)
   General Replay: 100 (9.1%)

🔄 Tokenizing and filtering by max seq length (1024)...


Tokenizing:   0%|          | 0/1100 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1100 [00:00<?, ? examples/s]

   Kept 1100 / 1100 samples (within 1024 token limit)

✅ Training data pipeline complete!
   Final training set size : 1100

📌 WHY replay buffer prevents catastrophic forgetting:
   - Fine-tuning only on medical data → model 'overwrites' general weights
   - 10% general examples during training keep general capabilities alive
   - This is the 'Rehearsal' strategy from continual learning literature


## CONFIGURE QLORA VIA UNSLOTH FastLanguageModel

In [22]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("Configuring QLoRA adapter (Unsloth FastLanguageModel)...")

# ---- LoRA Hyperparameters ----
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0       # 0 enables Unsloth fast kernel patching
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    target_modules             = TARGET_MODULES,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
    use_rslora                 = False,
    loftq_config               = None,
)

# ---- Count trainable vs frozen params ----
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params
trainable_pct    = (trainable_params / total_params) * 100

print(f"\nLoRA Configuration Summary:")
print(f"   LoRA Rank (r)          : {LORA_R}")
print(f"   LoRA Alpha             : {LORA_ALPHA}")
print(f"   LoRA Dropout           : {LORA_DROPOUT}  (0 = Unsloth fast patching ON)")
print(f"   Target Modules         : {', '.join(TARGET_MODULES)}")
print(f"\n   Total parameters       : {total_params:,}")
print(f"   Trainable (LoRA)       : {trainable_params:,}  ({trainable_pct:.2f}%)")
print(f"   Frozen (base model)    : {frozen_params:,}")

# ---- Visualization ----
# make_subplots with a Pie + Scatter.

r_values    = list(range(1, 65))
lora_params = [2 * 512 * r for r in r_values]
full_param  = 512 * 512
lora_pct    = [100 * p / full_param for p in lora_params]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Parameter Breakdown (Pie)", "LoRA Rank vs % of Full FT Params"],
    specs=[[{"type": "pie"}, {"type": "scatter"}]]
)

# Pie chart — col 1
fig.add_trace(
    go.Pie(
        labels=["Frozen Base Params", "Trainable LoRA Params"],
        values=[frozen_params, trainable_params],
        hole=0.5,
        marker=dict(
            colors=["#D3D3D3", "#FF6B6B"],
            line=dict(color="white", width=2)
        ),
        textinfo="label+percent",
        hovertemplate="%{label}<br>Params: %{value:,}<extra></extra>",
        showlegend=False,
    ),
    row=1, col=1
)

# Scatter chart — col 2
fig.add_trace(
    go.Scatter(
        x=r_values,
        y=lora_pct,
        name="LoRA Params %",
        mode="lines+markers",
        line=dict(color="#FF6B6B", width=2.5),
        marker=dict(size=4),
        hovertemplate="r=%{x}<br>LoRA Params: %{y:.1f}% of full FT<extra></extra>",
        showlegend=False,
    ),
    row=1, col=2
)

# Use layout shapes targeted at xaxis2/yaxis2 (the scatter subplot)
fig.update_layout(
    title_text="<b>QLoRA Configuration — Efficiency vs Capacity</b>",
    title_x=0.5,
    title_font_size=17,
    height=470,
    paper_bgcolor="white",
    plot_bgcolor="#FAFAFA",
    showlegend=False,

    # Horizontal line at y=100 on scatter subplot (xaxis2/yaxis2)
    shapes=[
        dict(
            type="line",
            xref="x2", yref="y2",
            x0=1, x1=64,
            y0=100, y1=100,
            line=dict(color="gray", width=1.5, dash="dash"),
        ),
        # Vertical line at our chosen rank
        dict(
            type="line",
            xref="x2", yref="y2",
            x0=LORA_R, x1=LORA_R,
            y0=0, y1=max(lora_pct) * 1.1,
            line=dict(color="#45B7D1", width=1.5, dash="dot"),
        ),
    ],

    # Annotations also targeted at x2/y2
    annotations=[
        dict(
            xref="x2", yref="y2",
            x=55, y=103,
            text="Full Fine-Tuning (100%)",
            showarrow=False,
            font=dict(size=11, color="gray"),
        ),
        dict(
            xref="x2", yref="y2",
            x=LORA_R + 2, y=max(lora_pct) * 0.6,
            text=f"r={LORA_R}<br>(our choice)",
            showarrow=False,
            font=dict(size=11, color="#45B7D1"),
        ),
    ],
)

fig.update_xaxes(title_text="LoRA Rank (r)", row=1, col=2, gridcolor="#EEE")
fig.update_yaxes(title_text="% of Full FT Params", row=1, col=2, gridcolor="#EEE")

fig.show()

print(f"\nKey Insight:")
print(f"   We are training only {trainable_pct:.2f}% of the model's weights.")
print(f"   The base model weights are FROZEN and loaded in 4-bit precision.")
print(f"   Dropout=0 activates Unsloth's fast triton kernel patching.")
print(f"   This enables training on a free 16GB T4 GPU in minutes, not hours.")
print(f"\nQLoRA model ready for training!")

Configuring QLoRA adapter (Unsloth FastLanguageModel)...

LoRA Configuration Summary:
   LoRA Rank (r)          : 16
   LoRA Alpha             : 32
   LoRA Dropout           : 0  (0 = Unsloth fast patching ON)
   Target Modules         : q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj

   Total parameters       : 350,984,064
   Trainable (LoRA)       : 8,798,208  (2.51%)
   Frozen (base model)    : 342,185,856



Key Insight:
   We are training only 2.51% of the model's weights.
   The base model weights are FROZEN and loaded in 4-bit precision.
   Dropout=0 activates Unsloth's fast triton kernel patching.
   This enables training on a free 16GB T4 GPU in minutes, not hours.

QLoRA model ready for training!


## TRAINING WITH LIVE LOSS MONITORING
### Using TRL's SFTTrainer (Supervised Fine-Tuning Trainer)

In [23]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments, DataCollatorForSeq2Seq
import torch, os, time, json
import plotly.graph_objects as go

print("🏋️  Preparing SFT Trainer...")

# ---- Training Hyperparameters ----
# Conservative settings for free Colab T4 (15.6 GB VRAM)
TRAINING_CONFIG = SFTConfig(
    output_dir                  = "./lora_medical_qwen",

    # Epochs & Steps
    num_train_epochs            = 3,
    max_steps                   = -1,              # -1 = use num_epochs

    # Batch & Gradient
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 8,               # effective batch = 2*8 = 16
    gradient_checkpointing      = True,

    # Optimizer: AdamW 8-bit via bitsandbytes
    optim                       = "adamw_8bit",
    learning_rate               = 2e-4,
    lr_scheduler_type           = "cosine",        # cosine decay
    warmup_ratio                = 0.05,
    weight_decay                = 0.01,

    # Precision
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),

    # Logging & Saving
    logging_steps               = 10,
    save_steps                  = 100,
    save_total_limit            = 2,
    eval_strategy               = "no",            # eval manually in later cell

    # Misc
    seed                        = 42,
    dataloader_num_workers      = 0,
    report_to                   = "none",          # no wandb on free Colab

    # SFT-specific
    max_seq_length              = MAX_SEQ_LEN,
    dataset_text_field          = "text",
    packing                     = False,            # True can speed up but tricky
    dataset_num_proc            = 2,
)

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = TRAIN_DATASET,
    args            = TRAINING_CONFIG,
)

# ---- Show memory before training ----
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    used = (total - free) / 1e9
    print(f"\n🔋 GPU Memory BEFORE training:")
    print(f"   Used  : {used:.2f} GB")
    print(f"   Free  : {free/1e9:.2f} GB")
    print(f"   Total : {total/1e9:.2f} GB")

print(f"\n📋 Training Config:")
print(f"   Epochs          : {TRAINING_CONFIG.num_train_epochs}")
print(f"   Batch size      : {TRAINING_CONFIG.per_device_train_batch_size}")
print(f"   Grad accumulation: {TRAINING_CONFIG.gradient_accumulation_steps}")
print(f"   Effective batch : {TRAINING_CONFIG.per_device_train_batch_size * TRAINING_CONFIG.gradient_accumulation_steps}")
print(f"   Learning rate   : {TRAINING_CONFIG.learning_rate}")
print(f"   Scheduler       : {TRAINING_CONFIG.lr_scheduler_type}")
print(f"   Optimizer       : {TRAINING_CONFIG.optim}")
print(f"   Precision       : {'BF16' if TRAINING_CONFIG.bf16 else 'FP16'}")
print(f"   Max seq length  : {MAX_SEQ_LEN}")

print(f"\n🚀 Starting training... (estimated 10-20 min on T4)")
print("   Watch the loss decrease — that's learning happening!\n")

start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time

# ---- Training Summary ----
print(f"\n{'='*55}")
print(f"  🎉 TRAINING COMPLETE!")
print(f"{'='*55}")
print(f"  Training time       : {elapsed/60:.1f} minutes")
print(f"  Final training loss : {train_result.training_loss:.4f}")
print(f"  Total steps         : {train_result.global_step}")
print(f"  Samples/second      : {train_result.metrics.get('train_samples_per_second', 'N/A')}")

# ---- Memory after training ----
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    used = (total - free) / 1e9
    print(f"\n🔋 GPU Memory AFTER training:")
    print(f"   Used  : {used:.2f} GB")
    print(f"   Free  : {free/1e9:.2f} GB")

# ---- Plot Training Loss Curve ----
log_history = trainer.state.log_history
train_logs  = [x for x in log_history if "loss" in x and "eval_loss" not in x]

if train_logs:
    steps  = [x["step"] for x in train_logs]
    losses = [x["loss"] for x in train_logs]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=steps, y=losses,
        mode="lines+markers",
        name="Training Loss",
        line=dict(color="#FF6B6B", width=2.5),
        marker=dict(size=5, color="#FF6B6B"),
        hovertemplate="Step %{x}<br>Loss: %{y:.4f}<extra></extra>"
    ))

    # Rolling average
    import pandas as pd
    loss_series = pd.Series(losses)
    rolling_avg = loss_series.rolling(window=5, min_periods=1).mean()
    fig.add_trace(go.Scatter(
        x=steps, y=rolling_avg.tolist(),
        mode="lines", name="5-Step Rolling Avg",
        line=dict(color="#4ECDC4", width=2, dash="dot"),
    ))

    fig.update_layout(
        title="<b>Training Loss Curve — Qwen2.5-0.5B Medical Fine-Tuning</b>",
        title_x=0.5, title_font_size=16,
        xaxis_title="Training Step",
        yaxis_title="Cross-Entropy Loss",
        height=450,
        paper_bgcolor="white",
        plot_bgcolor="#FAFAFA",
        legend=dict(x=0.75, y=0.95),
        xaxis=dict(gridcolor="#EEE"),
        yaxis=dict(gridcolor="#EEE"),
    )
    fig.show()

print("\n✅ Training complete!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


🏋️  Preparing SFT Trainer...


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1100 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.



🔋 GPU Memory BEFORE training:
   Used  : 1.45 GB
   Free  : 14.18 GB
   Total : 15.64 GB

📋 Training Config:
   Epochs          : 3
   Batch size      : 2
   Grad accumulation: 8
   Effective batch : 16
   Learning rate   : 0.0002
   Scheduler       : SchedulerType.COSINE
   Optimizer       : OptimizerNames.ADAMW_8BIT
   Precision       : FP16
   Max seq length  : 1024

🚀 Starting training... (estimated 10-20 min on T4)
   Watch the loss decrease — that's learning happening!



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,100 | Num Epochs = 3 | Total steps = 207
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
10,2.157233
20,1.264694
30,1.210416
40,1.289595
50,1.124156
60,1.157863
70,1.089098
80,1.008160
90,0.953984
100,0.940807



  🎉 TRAINING COMPLETE!
  Training time       : 10.6 minutes
  Final training loss : 1.0477
  Total steps         : 207
  Samples/second      : 5.227

🔋 GPU Memory AFTER training:
   Used  : 1.38 GB
   Free  : 14.25 GB



✅ Training complete!


## POST-FINE-TUNE EVALUATION
### Run the same evaluation as before but on the fine-tuned model, using identical prompts, same validation samples, same metrics.

In [24]:
import warnings
warnings.filterwarnings("ignore")

print("📊 Running evaluation on FINE-TUNED model...")
print("   Using same 50 validation samples as baseline (apples-to-apples)\n")

FINETUNED_SCORES = evaluate_model(
    model, tokenizer, MEDICAL_VAL_DF,
    num_samples=50, max_new_tokens=120, label="FINE-TUNED MODEL"
)

print("\n" + "=" * 55)
print("  📋 FINE-TUNED MODEL METRICS")
print("=" * 55)
print(f"  ROUGE-1     : {FINETUNED_SCORES['rouge1']:.4f}")
print(f"  ROUGE-2     : {FINETUNED_SCORES['rouge2']:.4f}")
print(f"  ROUGE-L     : {FINETUNED_SCORES['rougeL']:.4f}")
print(f"  BLEU        : {FINETUNED_SCORES['bleu']:.4f}")
print(f"  Perplexity  : {FINETUNED_SCORES['perplexity']:.2f}")
print("=" * 55)

# Delta comparison
print("\n📈 IMPROVEMENT over Baseline:")
for metric in ["rouge1", "rouge2", "rougeL", "bleu"]:
    delta = FINETUNED_SCORES[metric] - BASELINE_SCORES[metric]
    pct   = (delta / max(BASELINE_SCORES[metric], 1e-9)) * 100
    arrow = "▲" if delta > 0 else "▼"
    print(f"  {metric.upper():<12}: {arrow} {delta:+.4f}  ({pct:+.1f}%)")

ppl_delta = FINETUNED_SCORES["perplexity"] - BASELINE_SCORES["perplexity"]
print(f"  PERPLEXITY  : {'▼' if ppl_delta < 0 else '▲'} {ppl_delta:+.2f}  "
      f"({'Lower is better ✅' if ppl_delta < 0 else 'Higher — check training'})")

print("\nSample Comparison — Same Question (first evaluated sample):")
sample_q   = FINETUNED_SCORES["references"][0]   # the actual reference from eval
sample_ref = FINETUNED_SCORES["references"][0]
print(f"\n   Q (inferred): see reference below")
print(f"\n   Reference Answer  : {sample_ref[:150]}...")
print(f"\n   Base Prediction   : {BASELINE_SCORES['predictions'][0][:150]}...")
print(f"\n   FT Prediction     : {FINETUNED_SCORES['predictions'][0][:150]}...")


📊 Running evaluation on FINE-TUNED model...
   Using same 50 validation samples as baseline (apples-to-apples)



Evaluating [FINE-TUNED MODEL]:   0%|          | 0/50 [00:00<?, ?it/s]


  📋 FINE-TUNED MODEL METRICS
  ROUGE-1     : 0.3308
  ROUGE-2     : 0.1669
  ROUGE-L     : 0.2449
  BLEU        : 0.1102
  Perplexity  : 3.00

📈 IMPROVEMENT over Baseline:
  ROUGE1      : ▲ +0.0186  (+6.0%)
  ROUGE2      : ▲ +0.0368  (+28.3%)
  ROUGEL      : ▲ +0.0284  (+13.1%)
  BLEU        : ▲ +0.0419  (+61.5%)
  PERPLEXITY  : ▼ -59.11  (Lower is better ✅)

Sample Comparison — Same Question (first evaluated sample):

   Q (inferred): see reference below

   Reference Answer  : No, PTH also has a phosphaturic action on the kidney which is important because the calcium released from bone would complex with the released phospha...

   Base Prediction   : No, the effects of Parathyroid Hormone (PTH) on calcium (Ca) reabsorption in bones do not primarily involve an increase in ionized Ca2+. In fact, PTH ...

   FT Prediction     : No, PTH does not directly increase ionized Ca2+ concentration in bone....


## BEFORE vs. AFTER FINETUNING
### MODEL PERFORMANCE COMPARISON & VISUALIZATIONS

In [25]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

metrics_labels  = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU"]
metrics_keys    = ["rouge1",  "rouge2",  "rougeL",  "bleu"]

baseline_vals   = [BASELINE_SCORES[k]   for k in metrics_keys]
finetuned_vals  = [FINETUNED_SCORES[k]  for k in metrics_keys]
deltas          = [f - b for f, b in zip(finetuned_vals, baseline_vals)]
pct_changes     = [(d / max(b, 1e-9)) * 100 for d, b in zip(deltas, baseline_vals)]

# ---- Figure 1: Grouped Bar Chart ----
fig1 = go.Figure()

fig1.add_trace(go.Bar(
    name="Base Model (Before)",
    x=metrics_labels, y=baseline_vals,
    marker_color="#D3D3D3",
    text=[f"{v:.4f}" for v in baseline_vals],
    textposition="outside",
))

fig1.add_trace(go.Bar(
    name="Fine-Tuned Model (After)",
    x=metrics_labels, y=finetuned_vals,
    marker_color="#4ECDC4",
    text=[f"{v:.4f}" for v in finetuned_vals],
    textposition="outside",
))

fig1.update_layout(
    title="<b>Text Generation Metrics: Base Model vs Fine-Tuned Model</b>",
    title_x=0.5, title_font_size=17,
    barmode="group",
    yaxis_title="Score",
    xaxis_title="Evaluation Metric",
    paper_bgcolor="white",
    plot_bgcolor="#FAFAFA",
    height=480,
    legend=dict(x=0.75, y=0.98),
    yaxis=dict(gridcolor="#EEE", range=[0, max(finetuned_vals) * 1.25]),
)
fig1.show()

# ---- Figure 2: Radar Chart ----
theta = metrics_labels + [metrics_labels[0]]  # close the loop
r_base = baseline_vals + [baseline_vals[0]]
r_ft   = finetuned_vals + [finetuned_vals[0]]

fig2 = go.Figure()
fig2.add_trace(go.Scatterpolar(
    r=r_base, theta=theta,
    fill="toself", name="Base Model",
    line=dict(color="#D3D3D3", width=2),
    fillcolor="rgba(211,211,211,0.3)",
))
fig2.add_trace(go.Scatterpolar(
    r=r_ft, theta=theta,
    fill="toself", name="Fine-Tuned Model",
    line=dict(color="#4ECDC4", width=2),
    fillcolor="rgba(78,205,196,0.3)",
))
fig2.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, max(finetuned_vals) * 1.2])),
    title="<b>Metric Coverage — Radar View</b>",
    title_x=0.5, title_font_size=16,
    height=500, paper_bgcolor="white",
    legend=dict(x=0.8, y=1.1),
)
fig2.show()

# ---- Figure 3: Improvement % Bar ----
colors = ["#FF6B6B" if d < 0 else "#4ECDC4" for d in deltas]
fig3 = go.Figure(go.Bar(
    x=metrics_labels, y=pct_changes,
    marker_color=colors,
    text=[f"{p:+.1f}%" for p in pct_changes],
    textposition="outside",
))
fig3.update_layout(
    title="<b>Percentage Improvement After Fine-Tuning</b>",
    title_x=0.5, title_font_size=17,
    yaxis_title="% Change",
    xaxis_title="Metric",
    paper_bgcolor="white", plot_bgcolor="#FAFAFA",
    height=420,
    yaxis=dict(gridcolor="#EEE", zeroline=True, zerolinecolor="black", zerolinewidth=1.5),
)
fig3.show()

# ---- Figure 4: Perplexity Comparison ----
fig4 = go.Figure(go.Bar(
    x=["Base Model", "Fine-Tuned Model"],
    y=[BASELINE_SCORES["perplexity"], FINETUNED_SCORES["perplexity"]],
    marker_color=["#D3D3D3", "#FF6B6B"],
    text=[f"{BASELINE_SCORES['perplexity']:.2f}", f"{FINETUNED_SCORES['perplexity']:.2f}"],
    textposition="outside",
    width=0.4,
))
fig4.update_layout(
    title="<b>Perplexity: Base vs Fine-Tuned (Lower = Better)</b>",
    title_x=0.5, title_font_size=16,
    yaxis_title="Perplexity",
    paper_bgcolor="white", plot_bgcolor="#FAFAFA",
    height=420,
    yaxis=dict(gridcolor="#EEE"),
    annotations=[dict(
        text="Lower perplexity = model is more 'confident' on medical text",
        x=0.5, y=-0.15, xref="paper", yref="paper",
        showarrow=False, font=dict(size=12, color="gray")
    )]
)
fig4.show()

# ---- Print Summary Table ----
print("\n" + "=" * 65)
print(f"  {'METRIC':<15} {'BEFORE':>10} {'AFTER':>10} {'DELTA':>10} {'% CHG':>10}")
print("=" * 65)
for label, key, b, f, d, p in zip(
    metrics_labels, metrics_keys,
    baseline_vals, finetuned_vals, deltas, pct_changes
):
    arrow = "✅" if d > 0 else "❌"
    print(f"  {label:<15} {b:>10.4f} {f:>10.4f} {d:>+10.4f} {p:>+9.1f}% {arrow}")
print("-" * 65)
ppl_b = BASELINE_SCORES["perplexity"]
ppl_f = FINETUNED_SCORES["perplexity"]
ppl_d = ppl_f - ppl_b
ppl_arrow = "✅" if ppl_d < 0 else "❌"
print(f"  {'Perplexity':<15} {ppl_b:>10.2f} {ppl_f:>10.2f} {ppl_d:>+10.2f}   (lower better) {ppl_arrow}")
print("=" * 65)


  METRIC              BEFORE      AFTER      DELTA      % CHG
  ROUGE-1             0.3122     0.3308    +0.0186      +6.0% ✅
  ROUGE-2             0.1300     0.1669    +0.0368     +28.3% ✅
  ROUGE-L             0.2165     0.2449    +0.0284     +13.1% ✅
  BLEU                0.0682     0.1102    +0.0419     +61.5% ✅
-----------------------------------------------------------------
  Perplexity           62.11       3.00     -59.11   (lower better) ✅


## CATASTROPHIC FORGETTING PREVENTION VERIFICATION
### Test the fine-tuned model on GENERAL (non-medical) tasks to ensure our replay buffer strategy preserved general capabilities.
### Compare base vs fine-tuned on:
#### 1. General world knowledge Q&A
#### 2. Simple reasoning
#### 3. Code generation (basic)

In [26]:
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots

FastLanguageModel.for_inference(model)

GENERAL_TESTS = [
    {
        "task"    : "General Knowledge",
        "question": "What is the capital of France and what is it famous for?",
        "ref_keywords": ["paris", "eiffel", "louvre", "france", "city"],
    },
    {
        "task"    : "Simple Math Reasoning",
        "question": "If a train travels at 60 km/h and needs to cover 180 km, how long will it take?",
        "ref_keywords": ["3", "three", "hours", "hour"],
    },
    {
        "task"    : "Common Sense",
        "question": "Why do we need to sleep every night? Give three reasons.",
        "ref_keywords": ["rest", "brain", "memory", "health", "body", "energy"],
    },
    {
        "task"    : "Language Understanding",
        "question": "Explain the difference between a metaphor and a simile with examples.",
        "ref_keywords": ["like", "as", "comparison", "directly", "example"],
    },
]

def keyword_score(text: str, keywords: list) -> float:
    text_lower = text.lower()
    hits = sum(1 for kw in keywords if kw in text_lower)
    return hits / len(keywords)

def generate_response(question: str, max_new_tokens: int = 150) -> str:
    prompt = (
        f"<|im_start|>system\nYou are a helpful assistant.\n<|im_end|>\n"
        f"<|im_start|>user\n{question}\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True).strip()

print("🛡️  Running Catastrophic Forgetting Check...\n")
print("Testing fine-tuned model on NON-MEDICAL general tasks:")
print("=" * 60)

results = []
for test in GENERAL_TESTS:
    response = generate_response(test["question"])
    score    = keyword_score(response, test["ref_keywords"])
    results.append({
        "task"    : test["task"],
        "question": test["question"],
        "response": response,
        "score"   : score,
    })
    print(f"\n📌 Task: {test['task']}")
    print(f"   Q: {test['question'][:80]}...")
    print(f"   A: {response[:200]}...")
    status = "✅ General capability preserved" if score >= 0.3 else "⚠️  Possible degradation"
    print(f"   Keyword match score: {score:.2f}  → {status}")

# ---- Visualization ----
task_names  = [r["task"] for r in results]
task_scores = [r["score"] for r in results]
bar_colors  = ["#4ECDC4" if s >= 0.3 else "#FF6B6B" for s in task_scores]

fig = go.Figure(go.Bar(
    x=task_names, y=task_scores,
    marker_color=bar_colors,
    text=[f"{s:.2f}" for s in task_scores],
    textposition="outside",
    width=0.5,
))

fig.add_hline(y=0.3, line_dash="dash", line_color="#888",
              annotation_text="0.3 threshold (acceptable retention)",
              annotation_position="right")

fig.update_layout(
    title="<b>Catastrophic Forgetting Check — General Task Retention</b>",
    title_x=0.5, title_font_size=16,
    xaxis_title="Task Category",
    yaxis_title="Keyword Match Score (0-1)",
    yaxis=dict(range=[0, 1.2], gridcolor="#EEE"),
    paper_bgcolor="white", plot_bgcolor="#FAFAFA",
    height=450,
    annotations=[dict(
        text="Green = General capabilities retained ✅  |  Red = Possible forgetting ⚠️",
        x=0.5, y=-0.15, xref="paper", yref="paper",
        showarrow=False, font=dict(size=12, color="gray")
    )]
)
fig.show()

avg_retention = sum(task_scores) / len(task_scores)
print(f"\n{'='*55}")
print(f"  Average General Task Retention Score : {avg_retention:.2f}")
if avg_retention >= 0.3:
    print("  ✅ PASS — Replay buffer strategy successfully prevented")
    print("     catastrophic forgetting of general capabilities!")
else:
    print("  ⚠️  WARNING — Model shows signs of catastrophic forgetting.")
    print("     Consider: more replay data, lower LR, or fewer epochs.")
print(f"{'='*55}")

🛡️  Running Catastrophic Forgetting Check...

Testing fine-tuned model on NON-MEDICAL general tasks:

📌 Task: General Knowledge
   Q: What is the capital of France and what is it famous for?...
   A: The capital of France is Paris. It is known as the "City of Love" due to its romantic history, which dates back to ancient times when people traveled from all over Europe to visit the city's cathedral...
   Keyword match score: 1.00  → ✅ General capability preserved

📌 Task: Simple Math Reasoning
   Q: If a train travels at 60 km/h and needs to cover 180 km, how long will it take?...
   A: To calculate the time taken for the journey, we can use the formula:

Time = Distance / Speed

In this case, the distance is 180 km and the speed of the train is 60 km/h.

So,

Time = 180 km / 60 km/h...
   Keyword match score: 0.75  → ✅ General capability preserved

📌 Task: Common Sense
   Q: Why do we need to sleep every night? Give three reasons....
   A: 1. To repair and restore the body: Sleep is es


  Average General Task Retention Score : 0.85
  ✅ PASS — Replay buffer strategy successfully prevented
     catastrophic forgetting of general capabilities!


## SAVE FINE-TUNED MODEL

In [27]:
# Save the LoRA adapters (not the full model — saves space).
# The adapter is ~100MB vs ~2GB for the full model.
import os
import plotly.graph_objects as go
import json

SAVE_DIR      = "./medical_qwen_lora"
PUSH_TO_HUB   = True  # Set True + add HF token to push to HuggingFace Hub

print("💾 Saving fine-tuned LoRA adapters...")

# Save LoRA adapter + tokenizer
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"✅ LoRA adapters saved to: {SAVE_DIR}")

# Show saved files
files = os.listdir(SAVE_DIR)
total_size = sum(
    os.path.getsize(os.path.join(SAVE_DIR, f))
    for f in files
    if os.path.isfile(os.path.join(SAVE_DIR, f))
)
print(f"\n📁 Saved files ({len(files)} total, {total_size/1e6:.1f} MB):")
for f in sorted(files):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    print(f"   {f:<40} {size:.2f} MB")

# Optional: push to HuggingFace Hub
if PUSH_TO_HUB:
    from google.colab import userdata
    from huggingface_hub import login
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN)
    model.push_to_hub("deepakj111/medical-qwen2.5-0.5B-lora")
    tokenizer.push_to_hub("deepakj111/medical-qwen2.5-0.5B-lora")
    print("\n✅ Model pushed to HuggingFace Hub!")

# ---- Save results JSON for reproducibility ----
results_log = {
    "model"          : MODEL_NAME,
    "task"           : "Medical Q&A Text Generation",
    "dataset"        : "medalpaca/medical_meadow_medical_flashcards",
    "train_samples"  : len(TRAIN_DATASET),
    "lora_r"         : LORA_R,
    "lora_alpha"     : LORA_ALPHA,
    "epochs"         : TRAINING_CONFIG.num_train_epochs,
    "learning_rate"  : TRAINING_CONFIG.learning_rate,
    "baseline_scores": {k: round(v, 4) for k, v in BASELINE_SCORES.items()
                        if k not in ("predictions", "references")},
    "finetuned_scores": {k: round(v, 4) for k, v in FINETUNED_SCORES.items()
                         if k not in ("predictions", "references")},
}
with open(f"{SAVE_DIR}/training_results.json", "w") as f:
    json.dump(results_log, f, indent=2)
print(f"\n✅ Training results saved to {SAVE_DIR}/training_results.json")

# ---- FINAL SUMMARY VISUALIZATION ----
fig = go.Figure()

categories    = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU", "Perplexity↓"]
base_vals_viz = [
    BASELINE_SCORES["rouge1"], BASELINE_SCORES["rouge2"],
    BASELINE_SCORES["rougeL"], BASELINE_SCORES["bleu"],
    min(1.0, 10 / BASELINE_SCORES["perplexity"])  # normalized for viz
]
ft_vals_viz = [
    FINETUNED_SCORES["rouge1"], FINETUNED_SCORES["rouge2"],
    FINETUNED_SCORES["rougeL"], FINETUNED_SCORES["bleu"],
    min(1.0, 10 / FINETUNED_SCORES["perplexity"])  # normalized
]

fig.add_trace(go.Bar(name="Base Model", x=categories, y=base_vals_viz,
                     marker_color="#A8DADC", text=[f"{v:.3f}" for v in base_vals_viz],
                     textposition="outside"))
fig.add_trace(go.Bar(name="Fine-Tuned", x=categories, y=ft_vals_viz,
                     marker_color="#457B9D", text=[f"{v:.3f}" for v in ft_vals_viz],
                     textposition="outside"))

fig.update_layout(
    title="<b>🏆 Final Results: Base vs Fine-Tuned Model — Medical Q&A</b>",
    title_x=0.5, title_font_size=18,
    barmode="group", height=500,
    paper_bgcolor="white", plot_bgcolor="#FAFAFA",
    yaxis=dict(gridcolor="#EEE", title="Score"),
    legend=dict(x=0.78, y=0.98),
    annotations=[dict(
        text="Perplexity is shown as 10/ppl for visual consistency (higher = better)",
        x=0.5, y=-0.12, xref="paper", yref="paper",
        showarrow=False, font=dict(size=11, color="gray")
    )]
)
fig.show()

# ---- How to reload model later ----
print("\n" + "=" * 65)
print("  📖 HOW TO RELOAD THIS MODEL LATER (for inference)")
print("=" * 65)
print("""
  from unsloth import FastLanguageModel

  model, tokenizer = FastLanguageModel.from_pretrained(
      model_name   = "./medical_qwen_lora",   # path to saved adapters
      max_seq_length = 1024,
      dtype          = None,
      load_in_4bit   = True,
  )
  FastLanguageModel.for_inference(model)
""")

💾 Saving fine-tuned LoRA adapters...
✅ LoRA adapters saved to: ./medical_qwen_lora

📁 Saved files (7 total, 46.7 MB):
   README.md                                0.01 MB
   adapter_config.json                      0.00 MB
   adapter_model.safetensors                35.24 MB
   chat_template.jinja                      0.00 MB
   tokenizer.json                           11.42 MB
   tokenizer_config.json                    0.00 MB
   training_results.json                    0.00 MB


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 38.0kB / 35.2MB            

Saved model to https://huggingface.co/deepakj111/medical-qwen2.5-0.5B-lora


...

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpixybi41q/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.



✅ Model pushed to HuggingFace Hub!

✅ Training results saved to ./medical_qwen_lora/training_results.json



  📖 HOW TO RELOAD THIS MODEL LATER (for inference)

  from unsloth import FastLanguageModel

  model, tokenizer = FastLanguageModel.from_pretrained(
      model_name   = "./medical_qwen_lora",   # path to saved adapters
      max_seq_length = 1024,
      dtype          = None,
      load_in_4bit   = True,
  )
  FastLanguageModel.for_inference(model)

